In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd

# Load Dataset
DATA_PATH = "/content/drive/MyDrive/Weather Trend Forecasting/kaggle/GlobalWeatherRepository.csv"

df = pd.read_csv(DATA_PATH)

print("Original shape:", df.shape)
display(df.head())
print(df.info())

Mounted at /content/drive
Original shape: (141508, 41)


,country,location_name,latitude,longitude,timezone,last_updated_epoch,last_updated,temperature_celsius,temperature_fahrenheit,condition_text,...,air_quality_PM2.5,air_quality_PM10,air_quality_us-epa-index,air_quality_gb-defra-index,sunrise,sunset,moonrise,moonset,moon_phase,moon_illumination
0,Afghanistan,Kabul,34.52,69.18,Asia/Kabul,1715849100,2024-05-16 13:15,26.6,79.8,Partly Cloudy,...,8.4,26.6,1,1,04:50 AM,06:50 PM,12:12 PM,01:11 AM,Waxing Gibbous,55
1,Albania,Tirana,41.33,19.82,Europe/Tirane,1715849100,2024-05-16 10:45,19.0,66.2,Partly cloudy,...,1.1,2.0,1,1,05:21 AM,07:54 PM,12:58 PM,02:14 AM,Waxing Gibbous,55
2,Algeria,Algiers,36.76,3.05,Africa/Algiers,1715849100,2024-05-16 09:45,23.0,73.4,Sunny,...,10.4,18.4,1,1,05:40 AM,07:50 PM,01:15 PM,02:14 AM,Waxing Gibbous,55
3,Andorra,Andorra La Vella,42.50,1.52,Europe/Andorra,1715849100,2024-05-16 10:45,6.3,43.3,Light drizzle,...,0.7,0.9,1,1,06:31 AM,09:11 PM,02:12 PM,03:31 AM,Waxing Gibbous,55
4,Angola,Luanda,-8.84,13.23,Africa/Luanda,1715849100,2024-05-16 09:45,26.0,78.8,Partly cloudy,...,183.4,262.3,5,10,06:12 AM,05:55 PM,01:17 PM,12:38 AM,Waxing Gibbous,55


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 141508 entries, 0 to 141507
Data columns (total 41 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   country                       141508 non-null  object 
 1   location_name                 141508 non-null  object 
 2   latitude                      141508 non-null  float64
 3   longitude                     141508 non-null  float64
 4   timezone                      141508 non-null  object 
 5   last_updated_epoch            141508 non-null  int64  
 6   last_updated                  141508 non-null  object 
 7   temperature_celsius           141508 non-null  float64
 8   temperature_fahrenheit        141508 non-null  float64
 9   condition_text                141508 non-null  object 
 10  wind_mph                      141508 non-null  float64
 11  wind_kph                      141508 non-null  float64
 12  wind_degree                   141508 non-nul

In [ ]:
# Standardize Column Names and Parse Time
df.columns = df.columns.str.strip()

# Parse datetime
df["last_updated"] = pd.to_datetime(df["last_updated"], errors="coerce")

# Remove rows without essential identifiers/time
essential_cols = ["country", "location_name", "last_updated"]

before = len(df)
df = df.dropna(subset=essential_cols)
after = len(df)

print(f"Removed rows with missing essential fields: {before - after}")
print("Current shape:", df.shape)

Removed rows with missing essential fields: 0
Current shape: (141508, 41)


In [ ]:
# Missing Value Audit
missing_summary = (
    df.isna()
      .sum()
      .to_frame("missing_count")
)

missing_summary["missing_rate_%"] = 100 * missing_summary["missing_count"] / len(df)
missing_summary = missing_summary.sort_values("missing_rate_%", ascending=False)

display(missing_summary)

# Save missing report
OUTPUT_DIR = "/content/drive/MyDrive/Weather Trend Forecasting/processed_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

missing_summary.to_csv(os.path.join(OUTPUT_DIR, "missing_value_summary_before_cleaning.csv"))

,missing_count,missing_rate_%
country,0,0.0
location_name,0,0.0
latitude,0,0.0
longitude,0,0.0
timezone,0,0.0
last_updated_epoch,0,0.0
last_updated,0,0.0
temperature_celsius,0,0.0
temperature_fahrenheit,0,0.0
condition_text,0,0.0


In [ ]:
# Check pseudo-missing values
pseudo_missing_values = ["None", "none", "NULL", "null", "NaN", "nan", "N/A", "n/a",
                         "NA", "na", "Unknown", "unknown", "Missing", "missing",
                         "-", "--", ""]

pseudo_missing_summary = {}

for col in df.columns:
    pseudo_missing_summary[col] = df[col].astype(str).str.strip().isin(pseudo_missing_values).sum()

pseudo_missing_summary = pd.DataFrame.from_dict(
    pseudo_missing_summary,
    orient="index",
    columns=["pseudo_missing_count"]
)

pseudo_missing_summary["pseudo_missing_rate_%"] = (
    pseudo_missing_summary["pseudo_missing_count"] / len(df) * 100
)

pseudo_missing_summary = pseudo_missing_summary.sort_values(
    "pseudo_missing_rate_%",
    ascending=False
)

display(pseudo_missing_summary)

,pseudo_missing_count,pseudo_missing_rate_%
country,0,0.0
location_name,0,0.0
latitude,0,0.0
longitude,0,0.0
timezone,0,0.0
last_updated_epoch,0,0.0
last_updated,0,0.0
temperature_celsius,0,0.0
temperature_fahrenheit,0,0.0
condition_text,0,0.0


In [ ]:
# Define Feature Groups

geo_static_cols = [
    "country",
    "location_name",
    "latitude",
    "longitude",
    "timezone"
]

time_cols = [
    "last_updated_epoch",
    "last_updated"
]

temperature_cols = [
    "temperature_celsius",
    "temperature_fahrenheit",
    "feels_like_celsius",
    "feels_like_fahrenheit"
]

continuous_weather_cols = [
    "temperature_celsius",
    "temperature_fahrenheit",
    "wind_mph",
    "wind_kph",
    "wind_degree",
    "pressure_mb",
    "pressure_in",
    "precip_mm",
    "precip_in",
    "humidity",
    "cloud",
    "feels_like_celsius",
    "feels_like_fahrenheit",
    "visibility_km",
    "visibility_miles",
    "uv_index",
    "gust_mph",
    "gust_kph",
    "air_quality_Carbon_Monoxide",
    "air_quality_Ozone",
    "air_quality_Nitrogen_dioxide",
    "air_quality_Sulphur_dioxide",
    "air_quality_PM2.5",
    "air_quality_PM10",
    "air_quality_us-epa-index",
    "air_quality_gb-defra-index",
    "moon_illumination"
]

categorical_time_continuous_cols = [
    "condition_text",
    "wind_direction",
    "moon_phase"
]

astronomical_time_cols = [
    "sunrise",
    "sunset",
    "moonrise",
    "moonset"
]

# Keep only columns that actually exist
geo_static_cols = [c for c in geo_static_cols if c in df.columns]
continuous_weather_cols = [c for c in continuous_weather_cols if c in df.columns]
categorical_time_continuous_cols = [c for c in categorical_time_continuous_cols if c in df.columns]
astronomical_time_cols = [c for c in astronomical_time_cols if c in df.columns]

print("Geo/static columns:", geo_static_cols)
print("Continuous weather columns:", continuous_weather_cols)
print("Categorical columns:", categorical_time_continuous_cols)
print("Astronomical time columns:", astronomical_time_cols)

Geo/static columns: ['country', 'location_name', 'latitude', 'longitude', 'timezone']
Continuous weather columns: ['temperature_celsius', 'temperature_fahrenheit', 'wind_mph', 'wind_kph', 'wind_degree', 'pressure_mb', 'pressure_in', 'precip_mm', 'precip_in', 'humidity', 'cloud', 'feels_like_celsius', 'feels_like_fahrenheit', 'visibility_km', 'visibility_miles', 'uv_index', 'gust_mph', 'gust_kph', 'air_quality_Carbon_Monoxide', 'air_quality_Ozone', 'air_quality_Nitrogen_dioxide', 'air_quality_Sulphur_dioxide', 'air_quality_PM2.5', 'air_quality_PM10', 'air_quality_us-epa-index', 'air_quality_gb-defra-index', 'moon_illumination']
Categorical columns: ['condition_text', 'wind_direction', 'moon_phase']
Astronomical time columns: ['sunrise', 'sunset', 'moonrise', 'moonset']


In [ ]:
# Helper Functions

def fill_static_by_city(group, cols):
    """
    Fill static geographic fields within each city using city-level mode.
    """
    group = group.copy()
    for col in cols:
        if col in group.columns and group[col].isna().any():
            mode_value = group[col].mode(dropna=True)
            if len(mode_value) > 0:
                group[col] = group[col].fillna(mode_value.iloc[0])
    return group


def max_consecutive_missing(series):
    """
    Calculate maximum consecutive missing length in a series.
    """
    is_missing = series.isna().astype(int)
    if is_missing.sum() == 0:
        return 0
    groups = (is_missing != is_missing.shift()).cumsum()
    return is_missing.groupby(groups).sum().max()


def interpolate_numeric_by_time(group, numeric_cols, max_gap=6):
    """
    For each city:
    - Sort by time
    - Interpolate short missing gaps using time interpolation
    - Keep long missing gaps as NaN for later dropping
    """
    group = group.copy()
    group = group.sort_values("last_updated")
    group = group.set_index("last_updated")
    for col in numeric_cols:
        if col not in group.columns:
            continue

        # Only process numeric columns
        group[col] = pd.to_numeric(group[col], errors="coerce")

        # Interpolate only short gaps
        group[col] = group[col].interpolate(
            method="time",
            limit=max_gap,
            limit_direction="both"
        )
    group = group.reset_index()
    return group


def fill_categorical_by_city_time(group, cat_cols):
    """
    Fill weather category variables using forward/backward fill within each city.
    This preserves local temporal continuity better than global mode filling.
    """
    group = group.copy()
    group = group.sort_values("last_updated")
    for col in cat_cols:
        if col in group.columns:
            group[col] = group[col].ffill().bfill()
    return group


def fill_astronomical_time_cols(group, astro_cols):
    """
    Sunrise/sunset/moonrise/moonset are time-like strings.
    Use forward/backward fill within each city.
    """
    group = group.copy()
    group = group.sort_values("last_updated")
    for col in astro_cols:
        if col in group.columns:
            group[col] = group[col].ffill().bfill()
    return group

In [ ]:
# City-Level Sampling Interval Analysis

def city_time_diagnostics(df):
    records = []

    for city, g in df.groupby("location_name"):
        g = g.sort_values("last_updated")
        diffs = g["last_updated"].diff().dropna()

        if len(diffs) == 0:
            median_interval = pd.NaT
            mode_interval = pd.NaT
            irregular_ratio = np.nan
        else:
            median_interval = diffs.median()
            mode_interval = diffs.mode().iloc[0] if len(diffs.mode()) > 0 else pd.NaT
            irregular_ratio = (diffs != mode_interval).mean()

        records.append({
            "location_name": city,
            "country": g["country"].iloc[0],
            "record_count": len(g),
            "start_time": g["last_updated"].min(),
            "end_time": g["last_updated"].max(),
            "median_interval": median_interval,
            "mode_interval": mode_interval,
            "irregular_interval_ratio": irregular_ratio
        })

    return pd.DataFrame(records).sort_values("record_count", ascending=False)


city_diag = city_time_diagnostics(df)
display(city_diag.head(30))

city_diag.to_csv(os.path.join(OUTPUT_DIR, "city_time_sampling_diagnostics.csv"), index=False)

,location_name,country,record_count,start_time,end_time,median_interval,mode_interval,irregular_interval_ratio
240,Vatican City,Vatican City,728,2024-05-16 10:45:00,2026-05-15 08:30:00,1 days,1 days,0.599725
239,Valletta,Malta,728,2024-05-16 10:45:00,2026-05-15 08:30:00,1 days,1 days,0.585970
234,Tokyo,Japan,728,2024-05-16 17:45:00,2026-05-15 15:30:00,1 days,1 days,0.584594
228,Tashkent,Uzbekistan,728,2024-05-16 13:45:00,2026-05-15 11:45:00,1 days,1 days,0.572215
22,Asmara,Eritrea,728,2024-05-16 11:45:00,2026-05-15 09:30:00,1 days,1 days,0.562586
28,Baghdad,Iraq,728,2024-05-16 11:45:00,2026-05-15 09:30:00,1 days,1 days,0.577717
225,Suva,Fiji Islands,728,2024-05-16 20:45:00,2026-05-15 18:30:00,1 days,1 days,0.583219
44,Bern,Switzerland,728,2024-05-16 10:45:00,2026-05-15 08:45:00,1 days,1 days,0.595598
57,Bujumbura,Burundi,728,2024-05-16 10:45:00,2026-05-15 08:30:00,1 days,1 days,0.577717
116,Kyiv,Ukraine,728,2024-05-16 11:45:00,2026-05-15 09:45:00,1 days,1 days,0.581843


In [ ]:
# ============================================================
# 7. Missing Value Handling Pipeline
# ============================================================

df_clean = df.copy()

# Sort globally first
df_clean = df_clean.sort_values(["location_name", "last_updated"])

# Step 1: Fill static geographic fields by city
df_clean = (
    df_clean
    .groupby("location_name", group_keys=False)
    .apply(lambda g: fill_static_by_city(g, geo_static_cols))
)

# Step 2: Fill categorical weather descriptors by city-time order
df_clean = (
    df_clean
    .groupby("location_name", group_keys=False)
    .apply(lambda g: fill_categorical_by_city_time(g, categorical_time_continuous_cols))
)

# Step 3: Fill astronomical time-like fields by city-time order
df_clean = (
    df_clean
    .groupby("location_name", group_keys=False)
    .apply(lambda g: fill_astronomical_time_cols(g, astronomical_time_cols))
)

# Step 4: Interpolate continuous numerical weather variables by city and time
df_clean = (
    df_clean
    .groupby("location_name", group_keys=False)
    .apply(lambda g: interpolate_numeric_by_time(g, continuous_weather_cols, max_gap=6))
)

print("Shape after interpolation:", df_clean.shape)

/tmp/ipykernel_6087/1715946102.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: fill_static_by_city(g, geo_static_cols))
/tmp/ipykernel_6087/1715946102.py:21: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: fill_categorical_by_city_time(g, categorical_time_continuous_cols))
/tmp/ipykernel_6087/1715946102.py:28: DeprecationWarning: DataFrameGroupBy.apply operated on the gr

Shape after interpolation: (141508, 41)


/tmp/ipykernel_6087/1715946102.py:35: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: interpolate_numeric_by_time(g, continuous_weather_cols, max_gap=6))


In [ ]:
# Drop Remaining Missing Rows in Modeling-Critical Columns

critical_model_cols = [
    "temperature_celsius",
    "humidity",
    "pressure_mb",
    "wind_kph",
    "cloud",
    "precip_mm",
    "visibility_km"
]

critical_model_cols = [c for c in critical_model_cols if c in df_clean.columns]

before = len(df_clean)
df_clean = df_clean.dropna(subset=critical_model_cols)
after = len(df_clean)

print(f"Dropped rows with remaining missing values in critical modeling columns: {before - after}")
print("Cleaned shape:", df_clean.shape)

Dropped rows with remaining missing values in critical modeling columns: 0
Cleaned shape: (141508, 41)


In [ ]:
# ============================================================
# Missing Value Audit After Cleaning

missing_after = (
    df_clean.isna()
            .sum()
            .to_frame("missing_count")
)

missing_after["missing_rate_%"] = 100 * missing_after["missing_count"] / len(df_clean)
missing_after = missing_after.sort_values("missing_rate_%", ascending=False)

display(missing_after)

missing_after.to_csv(os.path.join(OUTPUT_DIR, "missing_value_summary_after_cleaning.csv"))

,missing_count,missing_rate_%
last_updated,0,0.0
country,0,0.0
location_name,0,0.0
latitude,0,0.0
longitude,0,0.0
timezone,0,0.0
last_updated_epoch,0,0.0
temperature_celsius,0,0.0
temperature_fahrenheit,0,0.0
condition_text,0,0.0


In [ ]:
# Save Cleaned Full Dataset

cleaned_path = os.path.join(OUTPUT_DIR, "GlobalWeatherRepository_missing_cleaned.csv")
df_clean.to_csv(cleaned_path, index=False)

print("Saved cleaned dataset to:")
print(cleaned_path)

Saved cleaned dataset to:
/content/drive/MyDrive/Weather Trend Forecasting/processed_outputs/GlobalWeatherRepository_missing_cleaned.csv


In [ ]:
# Save City-Level Cleaned Files

CITY_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "cleaned_by_city")
os.makedirs(CITY_OUTPUT_DIR, exist_ok=True)

for city, g in df_clean.groupby("location_name"):
    safe_city_name = (
        str(city)
        .replace("/", "_")
        .replace("\\", "_")
        .replace(" ", "_")
        .replace(",", "_")
    )

    city_file = os.path.join(CITY_OUTPUT_DIR, f"{safe_city_name}_cleaned.csv")
    g.sort_values("last_updated").to_csv(city_file, index=False)

print("Saved cleaned city-level datasets to:")
print(CITY_OUTPUT_DIR)

Saved cleaned city-level datasets to:
/content/drive/MyDrive/Weather Trend Forecasting/processed_outputs/cleaned_by_city


In [ ]:
# Save Top Complete Cities for Forecasting

city_counts_after = (
    df_clean.groupby(["country", "location_name"])
            .size()
            .reset_index(name="cleaned_record_count")
            .sort_values("cleaned_record_count", ascending=False)
)

display(city_counts_after.head(30))

city_counts_after.to_csv(
    os.path.join(OUTPUT_DIR, "city_record_counts_after_cleaning.csv"),
    index=False
)

,country,location_name,cleaned_record_count
0,Afghanistan,Kabul,728
47,Chad,N'djamena,728
206,Senegal,Dakar,728
36,Burundi,Bujumbura,728
231,Thailand,Nan,728
114,Jordan,Amman,728
105,Iraq,Baghdad,728
143,Malta,Valletta,728
71,Eritrea,Asmara,728
254,Uzbekistan,Tashkent,728


In [ ]:
# Save City-Level Cleaned Files with Strict City Name Cleaning

import os
import re
import unicodedata

CITY_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "cleaned_by_city_strict")
os.makedirs(CITY_OUTPUT_DIR, exist_ok=True)


def clean_city_filename(city_name):
    """
    Convert city name into a safe file name.
    Remove accents
    Replace all non-alphanumeric characters with underscores
    Remove duplicated underscores
    Remove leading/trailing underscores
    """
    city_name = str(city_name)

    # Normalize unicode characters, e.g., São Paulo -> Sao Paulo
    city_name = unicodedata.normalize("NFKD", city_name)
    city_name = city_name.encode("ascii", "ignore").decode("ascii")

    # Replace all non-letter/non-number characters with underscore
    city_name = re.sub(r"[^A-Za-z0-9]+", "_", city_name)

    # Remove duplicated underscores
    city_name = re.sub(r"_+", "_", city_name)

    # Remove leading/trailing underscores
    city_name = city_name.strip("_")

    # Fallback for empty names
    if city_name == "":
        city_name = "unknown_city"

    return city_name


used_filenames = set()

for city, g in df_clean.groupby("location_name"):
    safe_city_name = clean_city_filename(city)
    base_filename = f"{safe_city_name}_cleaned.csv"
    city_file = os.path.join(CITY_OUTPUT_DIR, base_filename)

    # Avoid overwriting if two city names become identical after cleaning
    counter = 1
    while city_file in used_filenames or os.path.exists(city_file):
        base_filename = f"{safe_city_name}_{counter}_cleaned.csv"
        city_file = os.path.join(CITY_OUTPUT_DIR, base_filename)
        counter += 1

    used_filenames.add(city_file)

    g.sort_values("last_updated").to_csv(city_file, index=False)

print("Saved cleaned city-level datasets to:")
print(CITY_OUTPUT_DIR)

Saved cleaned city-level datasets to:
/content/drive/MyDrive/Weather Trend Forecasting/processed_outputs/cleaned_by_city_strict
